In [1]:
import pandas as pd
import pickle
import json
from collections import Counter
import ast
import networkx as nx
from IPython.display import display

In [2]:
def string_to_list_conversion(str_list):
    """convert the string representation of the list to an actual list
    """
    return ast.literal_eval(str_list)

def load_pickle(file):
    """
    load a pickle file into a local variable
    can be a pandas dataframe or dictionary
    """
    pickle_in = open(file, "rb")
    pickle_file = pickle.load(pickle_in)
    pickle_in.close()
    return pickle_file

In [3]:
df = pd.read_csv('pass_paths.csv')
df['path'] = df['path'].apply(string_to_list_conversion)
node_dict = load_pickle('../node_dictionary.pickle')
# Use the map function to create a new 'layer_source' column in the dataframe
df['layer_source'] = df['source'].map({obj: color for color, obj_list in node_dict.items() for obj in obj_list})
df

,path,source,target,layer_source
0,"[APOE_A1, ANGULL02_FDG, MMSE]",APOE_A1,MMSE,GENETIC
1,"[APOE_A1, HMSCORE, MMSE]",APOE_A1,MMSE,GENETIC
2,"[APOE_A1, ANGULL01_FDG, APP, MMSE]",APOE_A1,MMSE,GENETIC
3,"[APOE_A1, ANGULL01_FDG, AXRASH, MMSE]",APOE_A1,MMSE,GENETIC
4,"[APOE_A1, ANGULL01_FDG, AXDROWSY, MMSE]",APOE_A1,MMSE,GENETIC
...,...,...,...,...
17872,"[MH18SURG, CINGPST07_FDG, AXDIZZY, UW_EF]",MH18SURG,UW_EF,RISKFACTORS
17873,"[MH18SURG, CINGPST07_FDG, AXSWEATN, UW_EF]",MH18SURG,UW_EF,RISKFACTORS
17874,"[MH18SURG, CINGPST07_FDG, AXWANDER, UW_EF]",MH18SURG,UW_EF,RISKFACTORS
17875,"[MH18SURG, CINGPST07_FDG, MH7DERM, UW_EF]",MH18SURG,UW_EF,RISKFACTORS


In [4]:
df['layer_source'].value_counts()

PET            8494
RISKFACTORS    4618
MRI            3226
MOLECULAR      1242
GENETIC         297
Name: layer_source, dtype: int64

In [5]:
# Veces que aparecen los nodos en los paths
nodos_unicos = df['path'].explode().unique()
count_nodo_por_layer_source = {}
for layer, grupo in df.groupby('layer_source'):
    count_nodo = {}
    for fila in grupo['path']:
        for nodo in fila:
            if nodo in count_nodo:
                count_nodo[nodo] += 1
            else:
                count_nodo[nodo] = 1
    count_nodo_por_layer_source[layer] = count_nodo

dataframes_por_layer = []
for layer, count_nodo in count_nodo_por_layer_source.items():
    df_nodos = pd.DataFrame({'nodos': list(count_nodo.keys()), 'count_nodo': list(count_nodo.values())})
    df_nodos['layer_source'] = layer
    dataframes_por_layer.append(df_nodos)

df_resultado = pd.concat(dataframes_por_layer)
df_resultado['layer'] = df_resultado['nodos'].map({obj: color for color, obj_list in node_dict.items() for obj in obj_list})
df_resultado.sort_values(by=['count_nodo'], ascending=False, inplace=True)
df_resultado

,nodos,count_nodo,layer_source,layer
4,AXDPMOOD,3268,PET,RISKFACTORS
1,AXENERGY,1790,RISKFACTORS,RISKFACTORS
48,AXDPMOOD,1401,MRI,RISKFACTORS
8,AXCRYING,1240,PET,RISKFACTORS
119,ANGULR04_FDG,1009,PET,PET
...,...,...,...,...
129,RH_SUPRAMARGINAL_AV45,1,MRI,PET
52,UPKelec_TAU,1,MRI,MOLECULAR
28,CEREB_WHITE,1,MOLECULAR,MRI
11,TEMPORAL_AV45,1,MOLECULAR,PET


In [16]:
df_resultado[(df_resultado.layer_source == 'MOLECULAR') & (df_resultado.layer == 'MOLECULAR')].head(5)

,nodos,count_nodo,layer_source,layer
53,APP,154,MOLECULAR,MOLECULAR
128,UPKelec_TAU,108,MOLECULAR,MOLECULAR
61,UPKelec_AB42,104,MOLECULAR,MOLECULAR
129,UPKelec_PTAU,100,MOLECULAR,MOLECULAR
77,TS_RATIO_ADJ,99,MOLECULAR,MOLECULAR


In this solution, we first create a list pairs that contains all pairs of consecutive nodes in the path column of the original dataframe. We do this by iterating over each row of the dataframe and over each element in the path list, and using the tuple function to create a pair of nodes.

We then use the Counter function from the Python standard library to count the number of times each pair appears in the pairs list.

Finally, we create a new dataframe df_pairs from the counts, by creating two columns: 'pair' with the pairs of nodes, and 'count' with the counts.

In [7]:
# create a list of all pairs of nodes in the paths
pairs = [tuple(df.loc[i, 'path'][j:j+2]) for i in range(len(df)) for j in range(len(df.loc[i, 'path'])-1)]
# count the pairs using Counter
pair_counts = Counter(pairs)

# create a new dataframe from the counts
df_pairs = pd.DataFrame({'pair': list(pair_counts.keys()), 'count': list(pair_counts.values())})
df_pairs['source'] = list(zip(*df_pairs['pair']))[0]
df_pairs['target'] = list(zip(*df_pairs['pair']))[1]
df_pairs

,pair,count,source,target
0,"(APOE_A1, ANGULL02_FDG)",8,APOE_A1,ANGULL02_FDG
1,"(ANGULL02_FDG, MMSE)",9,ANGULL02_FDG,MMSE
2,"(APOE_A1, HMSCORE)",11,APOE_A1,HMSCORE
3,"(HMSCORE, MMSE)",119,HMSCORE,MMSE
4,"(APOE_A1, ANGULL01_FDG)",33,APOE_A1,ANGULL01_FDG
...,...,...,...,...
3588,"(MH18SURG, CINGPST07_FDG)",115,MH18SURG,CINGPST07_FDG
3589,"(CINGPST07_FDG, AXFALL)",3,CINGPST07_FDG,AXFALL
3590,"(CINGPST07_FDG, AXINSOMN)",1,CINGPST07_FDG,AXINSOMN
3591,"(CINGPST07_FDG, UPK_AB42)",4,CINGPST07_FDG,UPK_AB42


In [8]:
# Crear el diccionario de path y pares
path_pairs = {}
for i in range(len(df)):
    pairs = [tuple(df.loc[i, 'path'][j:j+2]) for j in range(len(df.loc[i, 'path'])-1)]
    path_pairs[tuple(df.loc[i, 'path'])] = pairs

# # Mostrar el diccionario de pares
# for path, pairs in path_pairs.items():
#     print(f"Path: {path}")
#     print(f"Pares: {pairs}")
#     print()

new_df = pd.DataFrame({'path': key, 'pairs': value} for key, value in path_pairs.items())
new_df['sum_count'] = new_df['pairs'].apply(lambda pairs: df_pairs[df_pairs['pair'].isin(pairs)]['count'].sum())
new_df

,path,pairs,sum_count
0,"(APOE_A1, ANGULL02_FDG, MMSE)","[(APOE_A1, ANGULL02_FDG), (ANGULL02_FDG, MMSE)]",17
1,"(APOE_A1, HMSCORE, MMSE)","[(APOE_A1, HMSCORE), (HMSCORE, MMSE)]",130
2,"(APOE_A1, ANGULL01_FDG, APP, MMSE)","[(APOE_A1, ANGULL01_FDG), (ANGULL01_FDG, APP),...",76
3,"(APOE_A1, ANGULL01_FDG, AXRASH, MMSE)","[(APOE_A1, ANGULL01_FDG), (ANGULL01_FDG, AXRAS...",52
4,"(APOE_A1, ANGULL01_FDG, AXDROWSY, MMSE)","[(APOE_A1, ANGULL01_FDG), (ANGULL01_FDG, AXDRO...",57
...,...,...,...
17872,"(MH18SURG, CINGPST07_FDG, AXDIZZY, UW_EF)","[(MH18SURG, CINGPST07_FDG), (CINGPST07_FDG, AX...",142
17873,"(MH18SURG, CINGPST07_FDG, AXSWEATN, UW_EF)","[(MH18SURG, CINGPST07_FDG), (CINGPST07_FDG, AX...",146
17874,"(MH18SURG, CINGPST07_FDG, AXWANDER, UW_EF)","[(MH18SURG, CINGPST07_FDG), (CINGPST07_FDG, AX...",136
17875,"(MH18SURG, CINGPST07_FDG, MH7DERM, UW_EF)","[(MH18SURG, CINGPST07_FDG), (CINGPST07_FDG, MH...",130


Lo que queremos es separar según de qué capa es el source del path

In [9]:
layers = list(node_dict.keys())
del layers[-1]
all_nodes = []
for values in node_dict.values():
    all_nodes.extend(values)

for layer in layers:
    print(layer)
    df_temp = df[df.layer_source == layer].copy().reset_index(drop=True)

    pairs = [tuple(df_temp.loc[i, 'path'][j:j+2]) for i in range(len(df_temp)) for j in range(len(df_temp.loc[i, 'path'])-1)]
    pair_counts = Counter(pairs)

    df_pairs = pd.DataFrame({'pair': list(pair_counts.keys()), 'count': list(pair_counts.values())})
    df_pairs['source'] = list(zip(*df_pairs['pair']))[0]
    df_pairs['target'] = list(zip(*df_pairs['pair']))[1]

    path_pairs = {}
    for i in range(len(df)):
        pairs = [tuple(df.loc[i, 'path'][j:j+2]) for j in range(len(df.loc[i, 'path'])-1)]
        path_pairs[tuple(df.loc[i, 'path'])] = pairs
    new_df = pd.DataFrame({'path': key, 'pairs': value} for key, value in path_pairs.items())
    new_df['sum_count'] = new_df['pairs'].apply(lambda pairs: df_pairs[df_pairs['pair'].isin(pairs)]['count'].sum())
    new_df.sort_values(by=['sum_count'], ascending=False, inplace=True)
    display(new_df.head(10))

    G = nx.from_pandas_edgelist(df_pairs, source='source', target='target', edge_attr='count')
    G.add_nodes_from(all_nodes, ignore_existing=True) # add the rest of nodes without edges
    # G = nx.relabel_nodes(G, mapping)
    values = dict(G.degree)
    layers_net = {}
    for node in G:
        for layer_name in list(node_dict.keys()):
            if node in node_dict[layer_name]:
                layers_net[node] = layer_name
    nx.set_node_attributes(G, values, name='node_degree')
    nx.set_node_attributes(G, layers_net, name='layer')
    G_json = nx.readwrite.json_graph.cytoscape.cytoscape_data(G)
    json.dump(G_json, open(f'../web/archivos_redes/path_net_{layer}.json', 'w'))

GENETIC


,path,pairs,sum_count
42,"(APOE_A1, ANGULL01_FDG, AXRASH, ADAS13)","[(APOE_A1, ANGULL01_FDG), (ANGULL01_FDG, AXRAS...",44
31,"(APOE_A1, ANGULL01_FDG, AXRASH, ADSP_LAN)","[(APOE_A1, ANGULL01_FDG), (ANGULL01_FDG, AXRAS...",44
3,"(APOE_A1, ANGULL01_FDG, AXRASH, MMSE)","[(APOE_A1, ANGULL01_FDG), (ANGULL01_FDG, AXRAS...",44
37,"(APOE_A1, ANGULL01_FDG, AXRASH, ADSP_VSP)","[(APOE_A1, ANGULL01_FDG), (ANGULL01_FDG, AXRAS...",44
49,"(APOE_A1, ANGULL01_FDG, AXRASH, UW_EF)","[(APOE_A1, ANGULL01_FDG), (ANGULL01_FDG, AXRAS...",44
27,"(APOE_A1, ANGULL01_FDG, AXRASH, ADSP_EXF)","[(APOE_A1, ANGULL01_FDG), (ANGULL01_FDG, AXRAS...",44
7,"(APOE_A1, ANGULL01_FDG, AXRASH, MOCA)","[(APOE_A1, ANGULL01_FDG), (ANGULL01_FDG, AXRAS...",44
21,"(APOE_A1, ANGULL01_FDG, AXRASH, ADSP_MEM)","[(APOE_A1, ANGULL01_FDG), (ANGULL01_FDG, AXRAS...",44
45,"(APOE_A1, ANGULL01_FDG, AXRASH, UW_MEM)","[(APOE_A1, ANGULL01_FDG), (ANGULL01_FDG, AXRAS...",44
14,"(APOE_A1, ANGULL01_FDG, AXRASH, CDR)","[(APOE_A1, ANGULL01_FDG), (ANGULL01_FDG, AXRAS...",44


MOLECULAR


,path,pairs,sum_count
1335,"(UPKelec_TAU, ANGULL02_FDG, APP, ADSP_EXF)","[(UPKelec_TAU, ANGULL02_FDG), (ANGULL02_FDG, A...",129
1326,"(UPKelec_TAU, ANGULL02_FDG, APP, ADSP_MEM)","[(UPKelec_TAU, ANGULL02_FDG), (ANGULL02_FDG, A...",129
1301,"(UPKelec_TAU, ANGULL02_FDG, APP, MOCA)","[(UPKelec_TAU, ANGULL02_FDG), (ANGULL02_FDG, A...",129
1293,"(UPKelec_TAU, ANGULL02_FDG, APP, MMSE)","[(UPKelec_TAU, ANGULL02_FDG), (ANGULL02_FDG, A...",129
1351,"(UPKelec_TAU, ANGULL02_FDG, APP, ADSP_VSP)","[(UPKelec_TAU, ANGULL02_FDG), (ANGULL02_FDG, A...",129
1334,"(UPKelec_TAU, ANGULL02_FDG, TMPINFL04_FDG, ADS...","[(UPKelec_TAU, ANGULL02_FDG), (ANGULL02_FDG, T...",124
1325,"(UPKelec_TAU, ANGULL02_FDG, TMPINFL04_FDG, ADS...","[(UPKelec_TAU, ANGULL02_FDG), (ANGULL02_FDG, T...",124
1342,"(UPKelec_TAU, ANGULL02_FDG, TMPINFL04_FDG, ADS...","[(UPKelec_TAU, ANGULL02_FDG), (ANGULL02_FDG, T...",124
1300,"(UPKelec_TAU, ANGULL02_FDG, TMPINFL04_FDG, MOCA)","[(UPKelec_TAU, ANGULL02_FDG), (ANGULL02_FDG, T...",124
1318,"(UPKelec_TAU, ANGULL02_FDG, TMPINFL04_FDG, ADS...","[(UPKelec_TAU, ANGULL02_FDG), (ANGULL02_FDG, T...",124


PET


,path,pairs,sum_count
8875,"(CINGPSTR12_FDG, AXCRYING, AXDPMOOD, ADSP_EXF)","[(CINGPSTR12_FDG, AXCRYING), (AXCRYING, AXDPMO...",606
8891,"(CINGPSTR12_FDG, AXCRYING, AXDPMOOD, ADSP_VSP)","[(CINGPSTR12_FDG, AXCRYING), (AXCRYING, AXDPMO...",592
8883,"(CINGPSTR12_FDG, AXCRYING, AXDPMOOD, ADSP_LAN)","[(CINGPSTR12_FDG, AXCRYING), (AXCRYING, AXDPMO...",588
8927,"(CINGPSTR12_FDG, AXCRYING, AXDPMOOD, UW_EF)","[(CINGPSTR12_FDG, AXCRYING), (AXCRYING, AXDPMO...",586
8838,"(CINGPSTR12_FDG, AXCRYING, AXDPMOOD, MOCA)","[(CINGPSTR12_FDG, AXCRYING), (AXCRYING, AXDPMO...",585
8830,"(CINGPSTR12_FDG, AXCRYING, AXDPMOOD, MMSE)","[(CINGPSTR12_FDG, AXCRYING), (AXCRYING, AXDPMO...",584
8866,"(CINGPSTR12_FDG, AXCRYING, AXDPMOOD, ADSP_MEM)","[(CINGPSTR12_FDG, AXCRYING), (AXCRYING, AXDPMO...",583
8919,"(CINGPSTR12_FDG, AXCRYING, AXDPMOOD, UW_MEM)","[(CINGPSTR12_FDG, AXCRYING), (AXCRYING, AXDPMO...",576
8267,"(ANGULR05_FDG, AXCRYING, AXDPMOOD, ADSP_EXF)","[(ANGULR05_FDG, AXCRYING), (AXCRYING, AXDPMOOD...",562
8000,"(ANGULR02_FDG, AXCRYING, AXDPMOOD, ADSP_EXF)","[(ANGULR02_FDG, AXCRYING), (AXCRYING, AXDPMOOD...",559


MRI


,path,pairs,sum_count
12095,"(ST44CV, AXDPMOOD, GDS, MOCA)","[(ST44CV, AXDPMOOD), (AXDPMOOD, GDS), (GDS, MO...",278
12137,"(ST44CV, AXDPMOOD, GDS, ADSP_LAN)","[(ST44CV, AXDPMOOD), (AXDPMOOD, GDS), (GDS, AD...",276
12179,"(ST44CV, AXDPMOOD, GDS, UW_EF)","[(ST44CV, AXDPMOOD), (AXDPMOOD, GDS), (GDS, UW...",274
12172,"(ST44CV, AXDPMOOD, GDS, UW_MEM)","[(ST44CV, AXDPMOOD), (AXDPMOOD, GDS), (GDS, UW...",274
12121,"(ST44CV, AXDPMOOD, GDS, ADSP_MEM)","[(ST44CV, AXDPMOOD), (AXDPMOOD, GDS), (GDS, AD...",273
12089,"(ST44CV, AXDPMOOD, GDS, MMSE)","[(ST44CV, AXDPMOOD), (AXDPMOOD, GDS), (GDS, MM...",270
11798,"(ST32CV, AXDPMOOD, GDS, MOCA)","[(ST32CV, AXDPMOOD), (AXDPMOOD, GDS), (GDS, MO...",270
12129,"(ST44CV, AXDPMOOD, GDS, ADSP_EXF)","[(ST44CV, AXDPMOOD), (AXDPMOOD, GDS), (GDS, AD...",269
11847,"(ST32CV, AXDPMOOD, GDS, ADSP_LAN)","[(ST32CV, AXDPMOOD), (AXDPMOOD, GDS), (GDS, AD...",268
12145,"(ST44CV, AXDPMOOD, GDS, ADSP_VSP)","[(ST44CV, AXDPMOOD), (AXDPMOOD, GDS), (GDS, AD...",268


RISKFACTORS


,path,pairs,sum_count
15691,"(AXELMOOD, AXCRYING, AXDPMOOD, ADSP_MEM)","[(AXELMOOD, AXCRYING), (AXCRYING, AXDPMOOD), (...",390
15652,"(AXELMOOD, AXCRYING, AXDPMOOD, MMSE)","[(AXELMOOD, AXCRYING), (AXCRYING, AXDPMOOD), (...",387
15701,"(AXELMOOD, AXCRYING, AXDPMOOD, ADSP_EXF)","[(AXELMOOD, AXCRYING), (AXCRYING, AXDPMOOD), (...",386
15662,"(AXELMOOD, AXCRYING, AXDPMOOD, MOCA)","[(AXELMOOD, AXCRYING), (AXCRYING, AXDPMOOD), (...",386
15751,"(AXELMOOD, AXCRYING, AXDPMOOD, UW_MEM)","[(AXELMOOD, AXCRYING), (AXCRYING, AXDPMOOD), (...",384
15760,"(AXELMOOD, AXCRYING, AXDPMOOD, UW_EF)","[(AXELMOOD, AXCRYING), (AXCRYING, AXDPMOOD), (...",383
15711,"(AXELMOOD, AXCRYING, AXDPMOOD, ADSP_LAN)","[(AXELMOOD, AXCRYING), (AXCRYING, AXDPMOOD), (...",383
15721,"(AXELMOOD, AXCRYING, AXDPMOOD, ADSP_VSP)","[(AXELMOOD, AXCRYING), (AXCRYING, AXDPMOOD), (...",382
14319,"(AXHDACHE, AXCRYING, AXDPMOOD, ADSP_MEM)","[(AXHDACHE, AXCRYING), (AXCRYING, AXDPMOOD), (...",373
14282,"(AXHDACHE, AXCRYING, AXDPMOOD, MMSE)","[(AXHDACHE, AXCRYING), (AXCRYING, AXDPMOOD), (...",370
